# Sudan Climate Data Analysis

## Purpose
This notebook focuses specifically on Sudan's climate patterns as part of the regional analysis strategy.
**Why we're analyzing Sudan separately:**
- Sudan has unique desert and semi-desert climate characteristics
- Extreme temperature variations require isolated analysis
- Prevents skewed averages in multi-country datasets
- Enables targeted climate policy recommendations for Sudan's agricultural zones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Data Loading - Sudan Specific

**Intent:** Load only Sudan data to ensure focused analysis without cross-country contamination.

In [ ]:
# Load Sudan-specific climate data
df_sudan = pd.read_csv('../data/sudan.csv')

# Filter to ensure we only have Sudan data (quality check)
print(f"Dataset shape: {df_sudan.shape}")
print(f"Date range: {df_sudan['YEAR'].min()} to {df_sudan['YEAR'].max()}")
print(f"Total records: {len(df_sudan)}")

In [ ]:
# Data profiling for Sudan
print("=== SUDAN CLIMATE DATA PROFILING ===")
print("\nData Info:")
df_sudan.info()

print("\nDescriptive Statistics:")
print(df_sudan[['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']].describe())

## Data Cleaning - Sudan Specific

**Why we're cleaning Sudan data separately:**
- Sudan's desert climate may have different outlier patterns
- Extreme temperature variations require country-specific thresholds
- Ensures Sudan's climate patterns aren't masked by other countries

In [ ]:
# Check for missing values in Sudan data
print("Missing values in Sudan dataset:")
missing_values = df_sudan.isnull().sum()
print(missing_values[missing_values > 0])

# Check for invalid values (-999) common in climate data
invalid_counts = {}
for col in ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']:
    invalid_count = (df_sudan[col] == -999).sum()
    if invalid_count > 0:
        invalid_counts[col] = invalid_count

print(f"\nInvalid values (-999) in Sudan data: {invalid_counts}")

In [ ]:
# Clean Sudan data
df_sudan_clean = df_sudan.copy()

# Replace invalid values with NaN
climate_columns = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']
for col in climate_columns:
    df_sudan_clean[col] = df_sudan_clean[col].replace(-999, np.nan)

# Remove rows with critical missing data
initial_count = len(df_sudan_clean)
df_sudan_clean = df_sudan_clean.dropna(subset=['T2M', 'PRECTOTCORR'])
final_count = len(df_sudan_clean)

print(f"Sudan data cleaning results:")
print(f"Initial records: {initial_count}")
print(f"After removing invalid/critical missing data: {final_count}")
print(f"Records removed: {initial_count - final_count} ({((initial_count - final_count)/initial_count)*100:.2f}%)")

## Sudan Temperature Analysis

**Focus:** Understanding Sudan's extreme temperature patterns for agricultural and health policy planning.

In [ ]:
# Sudan temperature trends
plt.figure(figsize=(15, 10))

# Create date column for time series
df_sudan_clean['Date'] = pd.to_datetime(df_sudan_clean['YEAR'].astype(str) + '-' + 
                                       df_sudan_clean['DOY'].astype(str), format='%Y-%j')

# Plot 1: Daily Temperature Trends
plt.subplot(2, 2, 1)
plt.plot(df_sudan_clean['Date'], df_sudan_clean['T2M'], alpha=0.6, color='darkred')
plt.title('Sudan Daily Temperature Trends (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=45)

# Plot 2: Monthly Temperature Distribution
plt.subplot(2, 2, 2)
monthly_temp = df_sudan_clean.groupby(df_sudan_clean['Date'].dt.month)['T2M'].mean()
monthly_temp.plot(kind='bar', color='orange')
plt.title('Sudan Average Monthly Temperature')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

# Plot 3: Temperature Range Analysis (Sudan has extreme ranges)
plt.subplot(2, 2, 3)
temp_range = df_sudan_clean['T2M_MAX'] - df_sudan_clean['T2M_MIN']
plt.hist(temp_range.dropna(), bins=30, color='red', alpha=0.7)
plt.title('Sudan Daily Temperature Range Distribution')
plt.xlabel('Temperature Range (°C)')
plt.ylabel('Frequency')

# Plot 4: Yearly Temperature Trends
plt.subplot(2, 2, 4)
yearly_temp = df_sudan_clean.groupby('YEAR')['T2M'].mean()
plt.plot(yearly_temp.index, yearly_temp.values, marker='o', color='darkred')
plt.title('Sudan Yearly Average Temperature')
plt.xlabel('Year')
plt.ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

# Print key statistics
print("=== SUDAN TEMPERATURE ANALYSIS ===")
print(f"Average Temperature: {df_sudan_clean['T2M'].mean():.2f}°C")
print(f"Max Temperature: {df_sudan_clean['T2M_MAX'].max():.2f}°C")
print(f"Min Temperature: {df_sudan_clean['T2M_MIN'].min():.2f}°C")
print(f"Average Daily Range: {temp_range.mean():.2f}°C")
print(f"Max Daily Range: {temp_range.max():.2f}°C")

## Sudan Rainfall Analysis

**Purpose:** Analyze Sudan's limited rainfall patterns crucial for agricultural planning and water resource management.

In [ ]:
# Sudan rainfall analysis
plt.figure(figsize=(15, 10))

# Plot 1: Daily Rainfall
plt.subplot(2, 2, 1)
plt.plot(df_sudan_clean['Date'], df_sudan_clean['PRECTOTCORR'], alpha=0.6, color='blue')
plt.title('Sudan Daily Rainfall (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm/day)')
plt.xticks(rotation=45)

# Plot 2: Monthly Rainfall Patterns
plt.subplot(2, 2, 2)
monthly_rain = df_sudan_clean.groupby(df_sudan_clean['Date'].dt.month)['PRECTOTCORR'].mean()
monthly_rain.plot(kind='bar', color='cyan')
plt.title('Sudan Average Monthly Rainfall')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm/day)')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

# Plot 3: Rainfall Distribution
plt.subplot(2, 2, 3)
plt.hist(df_sudan_clean['PRECTOTCORR'].dropna(), bins=50, color='purple', alpha=0.7)
plt.title('Sudan Daily Rainfall Distribution')
plt.xlabel('Rainfall (mm/day)')
plt.ylabel('Frequency')
plt.yscale('log')  # Log scale due to many zero rainfall days

# Plot 4: Yearly Rainfall Totals
plt.subplot(2, 2, 4)
yearly_rain = df_sudan_clean.groupby('YEAR')['PRECTOTCORR'].sum()
plt.plot(yearly_rain.index, yearly_rain.values, marker='s', color='darkblue')
plt.title('Sudan Yearly Total Rainfall')
plt.xlabel('Year')
plt.ylabel('Total Rainfall (mm/year)')

plt.tight_layout()
plt.show()

# Print key statistics
print("=== SUDAN RAINFALL ANALYSIS ===")
print(f"Average Daily Rainfall: {df_sudan_clean['PRECTOTCORR'].mean():.2f} mm/day")
print(f"Max Daily Rainfall: {df_sudan_clean['PRECTOTCORR'].max():.2f} mm/day")
print(f"Rainy Days (>0.1mm): {(df_sudan_clean['PRECTOTCORR'] > 0.1).sum()}")
print(f"Dry Days: {(df_sudan_clean['PRECTOTCORR'] <= 0.1).sum()}")
print(f"Average Annual Rainfall: {yearly_rain.mean():.2f} mm/year")
print(f"Rainfall Seasonality: {monthly_rain.max()/monthly_rain.min():.2f}x variation")

## Sudan Climate Anomalies Detection

**Why this matters for Sudan:**
- Sudan's desert climate experiences extreme temperature anomalies
- Early detection supports climate adaptation planning
- Regional specificity improves policy relevance for Sudan's agriculture

In [ ]:
# Detect temperature anomalies in Sudan (using adjusted thresholds for desert climate)
temp_mean = df_sudan_clean['T2M'].mean()
temp_std = df_sudan_clean['T2M'].std()

# Define anomalies (±2.5 standard deviations for desert climate)
temp_upper_threshold = temp_mean + 2.5 * temp_std
temp_lower_threshold = temp_mean - 2.5 * temp_std

hot_days = df_sudan_clean[df_sudan_clean['T2M'] > temp_upper_threshold]
cold_days = df_sudan_clean[df_sudan_clean['T2M'] < temp_lower_threshold]

print("=== SUDAN TEMPERATURE ANOMALIES ===")
print(f"Temperature Mean: {temp_mean:.2f}°C")
print(f"Temperature Std Dev: {temp_std:.2f}°C")
print(f"Upper Threshold (±2.5σ): {temp_upper_threshold:.2f}°C")
print(f"Lower Threshold (±2.5σ): {temp_lower_threshold:.2f}°C")
print(f"\nExtreme Hot Days (>2.5σ): {len(hot_days)} ({len(hot_days)/len(df_sudan_clean)*100:.2f}%)")
print(f"Extreme Cold Days (<-2.5σ): {len(cold_days)} ({len(cold_days)/len(df_sudan_clean)*100:.2f}%)")

# Detect rainfall anomalies (important for Sudan's limited rainfall)
rain_mean = df_sudan_clean['PRECTOTCORR'].mean()
rain_std = df_sudan_clean['PRECTOTCORR'].std()

heavy_rain_threshold = rain_mean + 3 * rain_std  # Higher threshold for desert climate
heavy_rain_days = df_sudan_clean[df_sudan_clean['PRECTOTCORR'] > heavy_rain_threshold]

print(f"\n=== SUDAN RAINFALL ANOMALIES ===")
print(f"Rainfall Mean: {rain_mean:.2f} mm/day")
print(f"Heavy Rain Threshold (>3σ): {heavy_rain_threshold:.2f} mm/day")
print(f"Heavy Rain Events: {len(heavy_rain_days)} ({len(heavy_rain_days)/len(df_sudan_clean)*100:.2f}%)")

# Show dates of extreme events for policy planning
if len(hot_days) > 0:
    print(f"\nHottest Days in Sudan:")
    print(hot_days.nlargest(5, 'T2M')[['Date', 'T2M']].to_string(index=False))
    
if len(heavy_rain_days) > 0:
    print(f"\nHeaviest Rain Events in Sudan:")
    print(heavy_rain_days.nlargest(5, 'PRECTOTCORR')[['Date', 'PRECTOTCORR']].to_string(index=False))

## Sudan Climate Summary & Policy Insights

**Key findings for Sudan's climate policy planning:**

In [ ]:
# Generate Sudan-specific climate summary
sudan_summary = {
    'temperature_profile': {
        'mean_temp': df_sudan_clean['T2M'].mean(),
        'temp_variability': df_sudan_clean['T2M'].std(),
        'extreme_hot_days': len(hot_days),
        'extreme_cold_days': len(cold_days),
        'max_daily_range': temp_range.max()
    },
    'rainfall_profile': {
        'mean_daily_rainfall': df_sudan_clean['PRECTOTCORR'].mean(),
        'annual_total_rainfall': yearly_rain.mean(),
        'rainy_season_months': monthly_rain[monthly_rain > monthly_rain.mean()].index.tolist(),
        'heavy_rain_events': len(heavy_rain_days),
        'dry_season_length': (monthly_rain <= monthly_rain.mean()).sum()
    },
    'data_quality': {
        'total_records': len(df_sudan_clean),
        'data_completeness': (1 - df_sudan_clean.isnull().sum().sum() / (len(df_sudan_clean) * len(climate_columns))) * 100,
        'years_covered': df_sudan_clean['YEAR'].nunique()
    }
}

print("=== SUDAN CLIMATE SUMMARY FOR POLICY MAKERS ===")
for category, metrics in sudan_summary.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    for metric, value in metrics.items():
        if isinstance(value, float):
            print(f"  {metric.replace('_', ' ').title()}: {value:.2f}")
        else:
            print(f"  {metric.replace('_', ' ').title()}: {value}")

# Policy recommendations for Sudan
print("\n=== SUDAN-SPECIFIC POLICY RECOMMENDATIONS ===")
if sudan_summary['temperature_profile']['extreme_hot_days'] > len(df_sudan_clean) * 0.05:
    print("⚠️ EXTREME HEAT RISK: Implement heat wave early warning systems for urban areas")
if sudan_summary['temperature_profile']['max_daily_range'] > 20:
    print("⚠️ HIGH TEMPERATURE VARIABILITY: Develop climate-resilient agricultural practices")
if sudan_summary['rainfall_profile']['annual_total_rainfall'] < 500:
    print("⚠️ SEVERE WATER SCARCITY: Invest in drought-resistant crops and water conservation")
if sudan_summary['rainfall_profile']['dry_season_length'] > 8:
    print("⚠️ EXTENDED DRY SEASON: Implement water harvesting and storage systems")

print("\n✅ Sudan climate analysis completed successfully!")
print("Key insight: Sudan's extreme climate requires targeted adaptation strategies.")